# 基礎編３
このノートブックでは、スマートコントラクトをデプロイする一連のAPIの手順を例示します。

ここで示す手順の概略は下記のとおりです。
1. スマートコントラクトの新規作成
2. 引数型の設定
3. コードのデプロイ
4. 履歴開示先の設定

補足：  
他のサンプルコードで利用しているユーティリティ関数deploySmartContractには、同様の手順が含まれています。  
管理画面のGUI等を使ってスマートコントラクトをデプロイする場合にも、内部では同様の手順が実行されています。

## 準備
実行の準備を行います。

In [1]:
var { domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc } = await import('../lib/notebook-util.mjs');

## スマートコントラクトの新規作成

システムコントラクト'c1create'を使って、ブロックチェーン上に空のコントラクトを作成します。  
作成したコントラクトのIDを取得します。

In [2]:
var resp = await rpc.call(adminWallet, 'c1create', { type: 'contract', domain });
// if(resp.status !== 'ok') エラー処理をする
var newContractId = resp.value;
console.log('new contract ID:', newContractId);

new contract ID: c085697760


## 引数型の設定

システムコントラクト'c1update'を使って、作成したコントラクトに引数型を設定します。  
ここでは、一例として引数abcを数値型とし、引数xyzを文字列型として定義します。

In [3]:
var resp = await rpc.call(adminWallet, 'c1update', { id: newContractId, prop: 'argtypes', value: {
    abc: 'number', xyz: 'string'
} });
console.log(resp);

{
  txno: 701655,
  txid: 'xcDCsayM4hVaVa3dKcdF7xqkqs5UtdAaLzu7Ghx3oFft6',
  status: 'ok',
  value: null
}


## コードのデプロイ

システムコントラクト'c1update'を使って、作成したコントラクトにコードをデプロイします。  
ここでは、一例としてxyz+abcを返すコードをデプロイします。

In [4]:
var resp = await rpc.call(adminWallet, 'c1update', { id: newContractId, prop: 'code', value: 'return xyz+abc;' });
console.log(resp);

{
  txno: 701656,
  txid: 'xM8puTXj2jQSmFuHJ5xBayuYQNHYUWJBGfT83CZhMBmDPB',
  status: 'ok',
  value: null
}


## 履歴開示先の設定

システムコントラクト'c1update'を使って、作成したコントラクトの履歴開示先を設定します。  
ここでは、一例として全開示を指定します。

In [5]:
var resp = await rpc.call(adminWallet, 'c1update', { id: newContractId, prop: 'disclosed_to', value: ['all'] });
console.log(resp);

{
  txno: 701657,
  txid: 'xsXnE5ENsycTKwhoGWMjBL7nGnk9nCRTHLCdZaYTAwfFAB',
  status: 'ok',
  value: null
}


## 確認

確認のため、コントラクトの情報をブロックチェーンに問い合わせます。  

In [6]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'a_contract', id: newContractId });
console.log(resp.value);

{
  id: [ 'c085697760', 'c085697760@c.TDSL.H011' ],
  frozen: null,
  name: '',
  domain: [ 'd93319890', '@c.TDSL.H011' ],
  description: '',
  argtypes: { abc: 'number', xyz: 'string' },
  code: 'return xyz+abc;',
  mask: {},
  maxsteps: 0,
  a_txno: 701654,
  c_txno: 701654,
  num_transactions: 0,
  last_active: 1753420522857,
  created_at: 1753420522857,
  accessible_to: [ [ 'u58281888', 'jupyter@c.TDSL.H011' ] ],
  callable_to: [ 'all' ],
  disclosed_to: [ 'all' ],
  groups: [],
  managed: true,
  accessible: true
}


確認のため、新しいコントラクトのトランザクションを発行し、そのcalleeがnewContractIdと一致していることを確かめます。

In [7]:
var resp = await rpc.call(adminWallet, newContractId, { abc: 123, xyz: 'UVW' });
var retvalue = resp.value;
console.log('contract return value:', retvalue);
var resp = await rpc.call(adminWallet, 'c1query', { type: 'a_transaction', id: resp.txid });
if (resp.value.callee[0] === newContractId) console.log('OK'); else console.log('NG');
if (resp.value.valuestr === JSON.stringify(retvalue)) console.log('OK'); else console.log('NG');
console.log(resp.value);

contract return value: UVW123
OK
OK
{
  blkno: 73236,
  time: 1753420525540,
  txid: 'xEVGPeDAVFPoHrrrViJrzr9qQ7GSbDZUZKCDFha9VPYyHB',
  addr: 'eSQgkupyuMPkx22SuEui6HjUHyvTqm',
  caller: [ 'u58281888', 'jupyter@c.TDSL.H011' ],
  callee: [ 'c085697760', 'c085697760@c.TDSL.H011' ],
  status: 'ok',
  pack64: 'BQAAAAEAAADjAAAAAgAAAEAAAABAAAAAA3siY29udHJhY3QiOiJjMDg1Njk3NzYwIiwiYXJncyI6eyJhYmMiOjEyMywieHl6IjoiVVZXIn0sImFkZHIiOiJlU1Fna3VweXVNUGt4MjJTdUV1aTZIalVIeXZUcW0iLCJkZWFkbGluZSI6MTc1MzQyMDYyNTQyOCwib3JhY2xlIjoiUDR1Vmh0TnFZVUhKNXM5dyIsImJsb2NrcmVmIjp7Im5vIjo3MzIzNSwiaGFzaCI6Ikc4OE5mWHRWNmVRSStabDJXRjEyS0ZnTUdtQ0F6MTR3RUd2ajJZckN3UVE9In19ZXPxw2c37mebNx0UQDymdZOqotpzFW/twtDQIfAzQwYU8S32TPHQS3rATi17NeqF50z7jOEkewC5O78szMWyGk54T5DLJ1OUHFCflnRMeSvjmT0h3uTLxx/HQKfdLon+O0R+kNscnubM6uxhDfcMQ3Of5mjXwYj9Z6izsQDZSED91g==',
  txno: 701658,
  caller_txno: 0,
  argstr: '{"abc":123,"xyz":"UVW"}',
  valuestr: '"UVW123"',
  subtxs: 0,
  steps: 106,
  disclosed_to: [],
  related_to: [],
  cur_disclosed_t

## クリーンアップ
このノートブックの中で作成したコントラクトは今後使用しないので、削除しておきます。

In [8]:
var resp = await rpc.call(adminWallet, 'c1terminate', { id: newContractId });
console.log(resp);

{
  txno: 701659,
  txid: 'xfgAzJBPitrJ6UKzJgUPqGjE6xmMkNk8rnPviCjZMxPeQB',
  status: 'ok',
  value: 'c085697760'
}
